# Experimentos de detecção de veículos

Este notebook registra ground truth, benchmarks e visualizações. A aplicação final fica em `src/`.

A imagem usada é o JPEG extraído do PDF da prova (2048×1534 px). Não registre conclusões ou métricas sem executar as células correspondentes.

In [18]:
%matplotlib inline

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.widgets import RectangleSelector
from PIL import Image

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation.ground_truth import save_ground_truth  # noqa: E402

IMAGE_PATH = ROOT / "data/raw/drone_scene.jpg"
GROUND_TRUTH_PATH = ROOT / "data/annotations/ground_truth.json"
ANNOTATION_METHOD = "model_assisted_preannotations_manually_reviewed"
image = Image.open(IMAGE_PATH).convert("RGB")
image_width, image_height = image.size
print(f"Image: {image_width}x{image_height}")

Image: 2048x1534


## Anotação manual

Esta etapa é opcional: o ground truth revisado já está salvo no projeto. Para editar as caixas, execute `annotator = launch_manual_annotation()` em uma célula; o editor usa um widget interativo. Use `d` para apagar a última caixa e `s` para salvar. Antes de salvar, revise toda a rua e o estacionamento; cada veículo deve aparecer exatamente uma vez.

In [19]:
class ManualVehicleAnnotator:
    """Anotador interativo de retangulos para ground truth somente de veiculos."""

    def __init__(self, image: Image.Image, image_path: Path, output_path: Path) -> None:
        self.image = image
        self.image_path = image_path
        self.output_path = output_path
        self.boxes = self._load_existing()
        self.figure, self.axis = plt.subplots(figsize=(16, 12))
        self.axis.imshow(image)
        self.axis.set_title("Arraste: adiciona veiculo | botao direito: remove caixa | s: salva")
        self.axis.set_axis_off()
        self.selector = RectangleSelector(
            self.axis, self._on_select, useblit=True, button=[1], interactive=False
        )
        self.figure.canvas.mpl_connect("key_press_event", self._on_key)
        self.figure.canvas.mpl_connect("button_press_event", self._on_right_click)
        self._redraw()

    def _load_existing(self) -> list[list[float]]:
        if not self.output_path.exists():
            return []
        payload = json.loads(self.output_path.read_text(encoding="utf-8"))
        return [annotation["bbox_xyxy"] for annotation in payload["annotations"]]

    def _on_select(self, start, end) -> None:
        if None in (start.xdata, start.ydata, end.xdata, end.ydata):
            return
        x1, x2 = sorted((start.xdata, end.xdata))
        y1, y2 = sorted((start.ydata, end.ydata))
        if x2 - x1 >= 3 and y2 - y1 >= 3:
            self.boxes.append([round(x1, 2), round(y1, 2), round(x2, 2), round(y2, 2)])
            self._redraw()

    def _on_key(self, event) -> None:
        if event.key == "d" and self.boxes:
            self.boxes.pop()
            self._redraw()
        elif event.key == "s":
            self.save()

    def _on_right_click(self, event) -> None:
        if event.button != 3 or event.xdata is None or event.ydata is None:
            return
        for index in range(len(self.boxes) - 1, -1, -1):
            x1, y1, x2, y2 = self.boxes[index]
            if x1 <= event.xdata <= x2 and y1 <= event.ydata <= y2:
                self.boxes.pop(index)
                self._redraw()
                return

    def _redraw(self) -> None:
        for patch in list(self.axis.patches):
            patch.remove()
        for index, (x1, y1, x2, y2) in enumerate(self.boxes, start=1):
            self.axis.add_patch(
                Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec="#00e5ff", lw=1.5)
            )
            self.axis.text(
                x1, y1, str(index), color="white", fontsize=8, bbox={"facecolor": "#004d5c"}
            )
        self.axis.set_title(
            f"{len(self.boxes)} veiculos | arraste: adiciona | dir.: remove | d: ultima | s: salva"
        )
        self.figure.canvas.draw_idle()

    def save(self) -> None:
        save_ground_truth(
            self.output_path,
            self.image_path,
            self.boxes,
            annotation_method=ANNOTATION_METHOD,
            relative_to=ROOT,
        )
        print(f"Salvas {len(self.boxes)} anotacoes em {self.output_path.relative_to(ROOT)}")


def launch_manual_annotation(
    image_path: Path = IMAGE_PATH, output_path: Path = GROUND_TRUTH_PATH
) -> ManualVehicleAnnotator:
    """Abre o editor de caixas para qualquer imagem sem afetar os benchmarks."""
    get_ipython().run_line_magic("matplotlib", "widget")
    target_image = Image.open(image_path).convert("RGB")
    annotator = ManualVehicleAnnotator(target_image, image_path, output_path)
    plt.show()
    return annotator


print("Editor opcional: execute launch_manual_annotation() para revisar as caixas.")

Editor opcional: execute launch_manual_annotation() para revisar as caixas.


## Próximas células

Após concluir a revisão manual, este notebook carregará os detectores, executará os benchmarks controlados e apresentará as métricas e comparações visuais.

## Benchmarks reproduziveis

Execute as celulas abaixo somente apos revisar o ground truth. Cada configuracao usa a mesma imagem, o mesmo IoU de matching e tres repeticoes medidas depois de um warm-up.

In [20]:
%matplotlib inline

import numpy as np
import pandas as pd

from src.detection.sahi_detector import SahiVehicleDetector
from src.detection.yolo_detector import YoloVehicleDetector
from src.evaluation.benchmark import benchmark_detector
from src.evaluation.ground_truth import load_ground_truth
from src.roads.opencv_road import highlight_roads
from src.visualization.draw import (
    draw_detection_diagnostics,
    draw_detections,
    draw_ground_truth,
)

MODEL_DIR = ROOT / "models"
image_array = np.asarray(image)
ground_truth = load_ground_truth(GROUND_TRUTH_PATH)


def build_detector(
    model_name,
    domain,
    confidence=0.25,
    sahi_enabled=False,
    slice_size=512,
    image_size=1024,
    overlap=0.20,
):
    model_path = MODEL_DIR / model_name
    if not model_path.exists():
        raise FileNotFoundError(
            f"Model not found: {model_path}. Run scripts/download_models.py first."
        )
    if sahi_enabled:
        return SahiVehicleDetector(
            model_path,
            domain,
            confidence,
            image_size,
            slice_size,
            overlap,
            device="auto",
            merge_iou=0.50,
        )
    return YoloVehicleDetector(model_path, domain, confidence, image_size, 0.70, device="auto")


def run_benchmark_table(experiments):
    rows = []
    for name, detector_options in experiments:
        result = benchmark_detector(
            name, build_detector(**detector_options), image_array, ground_truth, repetitions=3
        )
        rows.append(result.as_dict())
    return pd.DataFrame(rows)


print(f"Ground truth reviewed: {len(ground_truth)} vehicles")

Ground truth reviewed: 55 vehicles


In [21]:
size_experiments = [
    ("aerial_n_normal", {"model_name": "yolo11n-obb.pt", "domain": "aerial"}),
    ("aerial_s_normal", {"model_name": "yolo11s-obb.pt", "domain": "aerial"}),
    ("aerial_m_normal", {"model_name": "yolo11m-obb.pt", "domain": "aerial"}),
]
size_results = run_benchmark_table(size_experiments)
size_results.sort_values(["f1", "median_inference_ms"], ascending=[False, True])

,name,predicted_count,true_positives,false_positives,false_negatives,precision,recall,f1,absolute_count_error,median_inference_ms,peak_gpu_memory_mb
0,aerial_n_normal,47,44,3,11,0.936170,0.8,0.862745,8,54.7885,551.562500
1,aerial_s_normal,48,44,4,11,0.916667,0.8,0.854369,7,48.2908,622.284668
2,aerial_m_normal,48,44,4,11,0.916667,0.8,0.854369,7,56.5561,754.634766


In [22]:
controlled_experiments = [
    ("coco_normal", {"model_name": "yolo11n.pt", "domain": "coco"}),
    ("coco_sahi_512", {"model_name": "yolo11n.pt", "domain": "coco", "sahi_enabled": True}),
    ("aerial_normal", {"model_name": "yolo11n-obb.pt", "domain": "aerial"}),
    ("aerial_sahi_512", {"model_name": "yolo11n-obb.pt", "domain": "aerial", "sahi_enabled": True}),
]
controlled_results = run_benchmark_table(controlled_experiments)
controlled_results.sort_values("f1", ascending=False)

,name,predicted_count,true_positives,false_positives,false_negatives,precision,recall,f1,absolute_count_error,median_inference_ms,peak_gpu_memory_mb
3,aerial_sahi_512,47,46,1,9,0.978723,0.836364,0.901961,8,961.6466,680.470215
2,aerial_normal,47,44,3,11,0.936170,0.800000,0.862745,8,52.8257,661.315430
0,coco_normal,1,0,1,55,0.000000,0.000000,0.000000,54,41.5537,649.459473
1,coco_sahi_512,4,0,4,55,0.000000,0.000000,0.000000,51,843.3305,670.074707


## Diagnostico de erros

Use esta visualizacao depois do benchmark 2x2 para revisar a configuracao candidata. Verde representa *true positive* (TP), amarelo *false positive* (FP) e vermelho *false negative* (FN), sempre usando o mesmo matching IoU >= 0.50 das metricas.

In [ ]:
diagnostic_detector = build_detector("yolo11n-obb.pt", "aerial", confidence=0.15, sahi_enabled=True)
diagnostic_detections = diagnostic_detector.predict(image_array)
diagnostic_image = draw_detection_diagnostics(
    image_array, diagnostic_detections, ground_truth, iou_threshold=0.50
)

figure, axis = plt.subplots(figsize=(14, 10))
axis.imshow(diagnostic_image)
axis.set_title("Detection audit: green=TP, amber=FP, red=FN")
axis.set_axis_off()
plt.tight_layout()
plt.show()
print("Green: TP | Amber: FP | Red: FN")

In [24]:
threshold_experiments = [
    (
        f"aerial_sahi_conf_{confidence:.2f}",
        {
            "model_name": "yolo11n-obb.pt",
            "domain": "aerial",
            "confidence": confidence,
            "sahi_enabled": True,
        },
    )
    for confidence in (0.15, 0.25, 0.35, 0.50)
]
threshold_results = run_benchmark_table(threshold_experiments)

slice_experiments = [
    (
        f"aerial_sahi_slice_{slice_size}",
        {
            "model_name": "yolo11n-obb.pt",
            "domain": "aerial",
            "sahi_enabled": True,
            "slice_size": slice_size,
        },
    )
    for slice_size in (512, 640)
]
slice_results = run_benchmark_table(slice_experiments)
display(threshold_results.sort_values("f1", ascending=False))
display(slice_results.sort_values("f1", ascending=False))

,name,predicted_count,true_positives,false_positives,false_negatives,precision,recall,f1,absolute_count_error,median_inference_ms,peak_gpu_memory_mb
1,aerial_sahi_conf_0.25,47,46,1,9,0.978723,0.836364,0.901961,8,932.5402,710.754395
0,aerial_sahi_conf_0.15,50,47,3,8,0.940000,0.854545,0.895238,5,991.7049,700.911133
2,aerial_sahi_conf_0.35,43,42,1,13,0.976744,0.763636,0.857143,12,876.5366,721.849121
3,aerial_sahi_conf_0.50,39,39,0,16,1.000000,0.709091,0.829787,16,838.3593,730.943848


,name,predicted_count,true_positives,false_positives,false_negatives,precision,recall,f1,absolute_count_error,median_inference_ms,peak_gpu_memory_mb
0,aerial_sahi_slice_512,47,46,1,9,0.978723,0.836364,0.901961,8,822.3556,736.926758
1,aerial_sahi_slice_640,47,44,3,11,0.936170,0.800000,0.862745,8,669.7983,747.021484


In [25]:
# One variable changes at a time relative to the 1024 px / 20% overlap control.
targeted_experiments = [
    (
        "aerial_sahi_control",
        {"model_name": "yolo11n-obb.pt", "domain": "aerial", "sahi_enabled": True},
    ),
    (
        "aerial_sahi_imgsz_1280",
        {
            "model_name": "yolo11n-obb.pt",
            "domain": "aerial",
            "sahi_enabled": True,
            "image_size": 1280,
        },
    ),
    (
        "aerial_sahi_overlap_030",
        {"model_name": "yolo11n-obb.pt", "domain": "aerial", "sahi_enabled": True, "overlap": 0.30},
    ),
]
targeted_results = run_benchmark_table(targeted_experiments)
targeted_results.sort_values("f1", ascending=False)

,name,predicted_count,true_positives,false_positives,false_negatives,precision,recall,f1,absolute_count_error,median_inference_ms,peak_gpu_memory_mb
0,aerial_sahi_control,47,46,1,9,0.978723,0.836364,0.901961,8,868.4551,578.424316
2,aerial_sahi_overlap_030,47,44,3,11,0.936170,0.800000,0.862745,8,1049.5528,598.114746
1,aerial_sahi_imgsz_1280,39,37,2,18,0.948718,0.672727,0.787234,16,951.4009,621.266602


In [ ]:
final_detector = build_detector("yolo11n-obb.pt", "aerial", confidence=0.15, sahi_enabled=True)
final_detections = final_detector.predict(image_array)

figure, axes = plt.subplots(1, 3, figsize=(22, 8))
for axis, title, rendered in zip(
    axes,
    ("Original", "Ground truth reviewed", "Final prediction"),
    (
        image_array,
        draw_ground_truth(image_array, ground_truth),
        draw_detections(image_array, final_detections),
    ),
    strict=True,
):
    axis.imshow(rendered)
    axis.set_title(title)
    axis.set_axis_off()
plt.tight_layout()
print(f"Final predicted count: {len(final_detections)}")

## Laboratorio da malha viaria

Este experimento permanece somente no notebook. Ele compara o destaque HSV atual com uma hipotese classica: vias tendem a formar faixas longas horizontais ou verticais, portanto uma abertura morfologica retangular pode reduzir regioes pequenas. Nao e uma segmentacao semantica nem uma configuracao da aplicacao final.

In [ ]:
import cv2


def overlay_mask(image, mask, color=(242, 107, 56), opacity=0.35):
    rendered = image.copy()
    selected = mask > 0
    rendered[selected] = (image[selected] * (1 - opacity) + np.asarray(color) * opacity).astype(
        np.uint8
    )
    return rendered


def linear_road_candidate(image):
    # Same HSV candidate as the app; only the geometric post-processing changes.
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    candidate = cv2.inRange(
        hsv, np.asarray([5, 35, 45], dtype=np.uint8), np.asarray([35, 95, 190], dtype=np.uint8)
    )
    base_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    cleaned = cv2.morphologyEx(candidate, cv2.MORPH_CLOSE, base_kernel)
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_OPEN, base_kernel)

    horizontal = cv2.morphologyEx(
        cleaned, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (81, 9))
    )
    vertical = cv2.morphologyEx(
        cleaned, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (9, 81))
    )
    linear = cv2.bitwise_or(horizontal, vertical)
    return cv2.morphologyEx(
        linear,
        cv2.MORPH_CLOSE,
        cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (17, 17)),
    )


baseline_roads = highlight_roads(
    image_array, hsv_lower=(5, 35, 45), hsv_upper=(35, 95, 190), kernel_size=7, min_area=1500
)
linear_mask = linear_road_candidate(image_array)

figure, axes = plt.subplots(2, 2, figsize=(18, 14))
views = [
    ("Original", image_array),
    ("Vehicle detections (final configuration)", draw_detections(image_array, final_detections)),
    ("Road baseline: HSV + morphology", baseline_roads.overlay),
    ("Experiment: HSV + linear continuity", overlay_mask(image_array, linear_mask)),
]
for axis, (title, rendered) in zip(axes.flat, views, strict=True):
    axis.imshow(rendered)
    axis.set_title(title)
    axis.set_axis_off()
plt.tight_layout()
plt.show()

for name, mask in [("baseline", baseline_roads.mask), ("linear experiment", linear_mask)]:
    coverage = 100 * np.count_nonzero(mask) / mask.size
    print(f"{name}: {coverage:.2f}% of image highlighted")

## Experimento neural: segmentacao semantica de vias

A mascara HSV anterior confunde pavimento com telhados e solo exposto porque usa somente cor. Nesta etapa testamos, **somente no notebook**, o checkpoint [`mfaytin/mask2former-satellite`](https://huggingface.co/mfaytin/mask2former-satellite), um Mask2Former treinado no OpenEarthMap para classes de cobertura do solo. O objetivo e verificar se contexto visual (rua, vegetacao, construcao e solo) reduz esses falsos positivos.

O modelo nao foi treinado nesta imagem e nenhum fine-tuning sera feito agora. Ha diferenca entre a resolucao/origem do treino e a foto de drone; portanto a comparacao e qualitativa e nao autoriza, por si so, alterar o pipeline final.

### 1. Carregar checkpoint e imagem

Esta celula usa CUDA quando disponivel, baixa os pesos apenas na primeira execucao e armazena o cache em `cache/huggingface/` (ignorado pelo Git). O checkpoint possui IDs genericos (`LABEL_n`) no arquivo de configuracao; a proxima celula mostra todas as classes preditas para validar visualmente qual ID corresponde a via nesta imagem.

In [28]:
from time import perf_counter

import torch
from PIL import Image
from transformers import Mask2FormerForUniversalSegmentation, Mask2FormerImageProcessor

SEGMENTATION_MODEL_ID = "mfaytin/mask2former-satellite"
SEGMENTATION_ROAD_CLASS_ID = 3
SEGMENTATION_CACHE_DIR = ROOT / "cache" / "huggingface"
SEGMENTATION_CACHE_DIR.mkdir(parents=True, exist_ok=True)
SEGMENTATION_DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"


def neural_overlay(image, mask, color=(242, 107, 56), opacity=0.40):
    rendered = image.copy()
    selected = mask > 0
    rendered[selected] = (image[selected] * (1 - opacity) + np.asarray(color) * opacity).astype(
        np.uint8
    )
    return rendered


segmentation_processor = Mask2FormerImageProcessor.from_pretrained(
    SEGMENTATION_MODEL_ID, cache_dir=SEGMENTATION_CACHE_DIR
)
segmentation_model = Mask2FormerForUniversalSegmentation.from_pretrained(
    SEGMENTATION_MODEL_ID, cache_dir=SEGMENTATION_CACHE_DIR
).to(SEGMENTATION_DEVICE)
segmentation_model.eval()
segmentation_input = Image.fromarray(image_array)

print(f"Segmentation device: {SEGMENTATION_DEVICE}")
print(f"Model: {SEGMENTATION_MODEL_ID}")
print("Run the class audit below before interpreting the road mask.")

Segmentation device: cuda:0
Model: mfaytin/mask2former-satellite
Run the class audit below before interpreting the road mask.


### 2. Inferir e auditar os IDs de classe

O mapa e reamostrado para as dimensoes originais da imagem. Como o checkpoint nao fornece nomes confiaveis para os IDs, esta celula destaca cada classe predita com uma cor diferente. Nesta cena, a auditoria visual identificou `LABEL_3` como a via: ela acompanha a rua central e os acessos, enquanto `LABEL_4` acompanha arvores. Mantenha essa celula como evidencia da decisao; nao presuma que o mesmo ID sera valido para checkpoints diferentes.

In [ ]:
segmentation_inputs = segmentation_processor(images=segmentation_input, return_tensors="pt")
segmentation_inputs = {
    name: value.to(SEGMENTATION_DEVICE) for name, value in segmentation_inputs.items()
}
started_at = perf_counter()
with torch.inference_mode():
    segmentation_outputs = segmentation_model(**segmentation_inputs)
segmentation_ms = (perf_counter() - started_at) * 1_000
segmentation_map = (
    segmentation_processor.post_process_semantic_segmentation(
        segmentation_outputs, target_sizes=[image_array.shape[:2]]
    )[0]
    .detach()
    .cpu()
    .numpy()
)

audit_colors = [(0, 140, 255), (255, 193, 7), (0, 200, 83), (244, 67, 54), (156, 39, 176)]
predicted_class_ids = np.unique(segmentation_map)
figure, axes = plt.subplots(1, len(predicted_class_ids), figsize=(5 * len(predicted_class_ids), 5))
for index, (axis, class_id) in enumerate(zip(axes, predicted_class_ids, strict=True)):
    class_mask = segmentation_map == class_id
    rendered = neural_overlay(
        image_array,
        class_mask.astype(np.uint8) * 255,
        color=audit_colors[index % len(audit_colors)],
        opacity=0.45,
    )
    axis.imshow(rendered)
    axis.set_title(f"LABEL_{class_id}: {100 * class_mask.mean():.1f}%")
    axis.set_axis_off()
plt.tight_layout()
plt.show()
print(f"Segmentation inference: {segmentation_ms:.1f} ms")
print(f"Predicted class IDs: {predicted_class_ids.tolist()}")

### 3. Comparar veiculos, HSV e segmentacao

Execute antes a celula `final-visual-comparison`, que produz `final_detections`. Esta comparacao coloca a previsao final de veiculos ao lado da camada HSV atual e da mascara neural. Revise principalmente: (1) a rua principal e o cruzamento devem estar destacados; (2) telhados, areia e barro nao devem dominar a mascara; e (3) o resultado ainda pode conter pequenos falsos positivos e trechos ausentes.

In [ ]:
neural_road_mask = (segmentation_map == SEGMENTATION_ROAD_CLASS_ID).astype(np.uint8) * 255
neural_road_overlay = neural_overlay(image_array, neural_road_mask, opacity=0.40)
neural_road_coverage = 100 * np.count_nonzero(neural_road_mask) / neural_road_mask.size
neural_baseline_roads = highlight_roads(
    image_array, hsv_lower=(5, 35, 45), hsv_upper=(35, 95, 190), kernel_size=7, min_area=1500
)
baseline_road_coverage = (
    100 * np.count_nonzero(neural_baseline_roads.mask) / neural_baseline_roads.mask.size
)

figure, axes = plt.subplots(1, 4, figsize=(26, 7))
views = [
    ("Original", image_array),
    ("Vehicle detections (final configuration)", draw_detections(image_array, final_detections)),
    ("Road baseline: HSV + morphology", neural_baseline_roads.overlay),
    ("Road experiment: Mask2Former LABEL_3", neural_road_overlay),
]
for axis, (title, rendered) in zip(axes, views, strict=True):
    axis.imshow(rendered)
    axis.set_title(title)
    axis.set_axis_off()
plt.tight_layout()
plt.show()
print(f"Mask2Former road coverage: {neural_road_coverage:.2f}% of image")
print(f"HSV baseline coverage: {baseline_road_coverage:.2f}% of image")
print("Qualitative result only: no road ground truth or IoU is available yet.")

### Leitura preliminar desta imagem

No ambiente local, a mascara neural `LABEL_3` cobriu 5,59% da imagem, contra 34,91% da mascara HSV atual. Na revisao visual, ela recuperou a rua principal, o cruzamento e acessos relevantes sem classificar telhados e grandes areas de barro como via. Ainda existem pequenos falsos positivos e possiveis trechos ausentes.

Esse e um resultado qualitativo de uma unica imagem. Antes de integrar o modelo ao pipeline, a proxima evidencia necessaria e um ground truth de via em imagens independentes para calcular IoU, precision e recall de segmentacao.

## Avaliacao externa: veiculos quantitativos e vias qualitativas

As tres imagens diurnas selecionadas em `data/external/` verificam se as duas tarefas funcionam fora da imagem do desafio. Elas nao participam da escolha de modelo, threshold ou slice size: esses parametros permanecem congelados. A imagem noturna e mantida somente como teste qualitativo de cenario adverso e fica fora das metricas agregadas.

A avaliacao de veiculos e quantitativa apos a revisao humana do ground truth. A avaliacao de malha viaria e visual/qualitativa, pois este conjunto nao possui mascaras de estrada. Utilize somente imagens cuja origem e licenca estejam registradas antes de versionar ou apresentar os resultados.

### 1. Descobrir as imagens locais

A celula valida somente formatos de imagem suportados e mostra os nomes/dimensoes que serao analisados. Ela falha de forma explicita se a pasta estiver vazia, evitando resultados silenciosos.

In [31]:
EXTERNAL_IMAGES_DIR = ROOT / "data" / "external"
EXTERNAL_ANNOTATIONS_DIR = ROOT / "data" / "annotations" / "external"
EXTERNAL_IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp"}
all_external_image_paths = sorted(
    path
    for path in EXTERNAL_IMAGES_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in EXTERNAL_IMAGE_SUFFIXES
)
EXTERNAL_QUALITATIVE_ONLY_FILES = {"testeviario4.jpg"}
external_image_paths = [
    path for path in all_external_image_paths if path.name not in EXTERNAL_QUALITATIVE_ONLY_FILES
]
qualitative_only_image_paths = [
    path for path in all_external_image_paths if path.name in EXTERNAL_QUALITATIVE_ONLY_FILES
]
if not external_image_paths:
    raise FileNotFoundError(f"No supported quantitative images found in {EXTERNAL_IMAGES_DIR}")


def external_annotation_path(image_path: Path) -> Path:
    """Retorna o JSON de anotacoes associado a uma imagem externa."""
    return EXTERNAL_ANNOTATIONS_DIR / f"{image_path.stem}.json"


external_image_inventory = []
for path in all_external_image_paths:
    with Image.open(path) as external_image:
        external_image_inventory.append(
            {
                "file": path.name,
                "width": external_image.width,
                "height": external_image.height,
                "usage": (
                    "qualitative stress test"
                    if path.name in EXTERNAL_QUALITATIVE_ONLY_FILES
                    else "quantitative evaluation"
                ),
            }
        )
display(pd.DataFrame(external_image_inventory))
print(f"External images selected for metrics: {len(external_image_paths)}")
if qualitative_only_image_paths:
    print(
        "Qualitative-only images: " + ", ".join(path.name for path in qualitative_only_image_paths)
    )

,file,width,height,usage
0,testeviario1.jpg,1152,641,quantitative evaluation
1,testeviario2.jpg,1151,647,quantitative evaluation
2,testeviario3.jpg,1160,647,quantitative evaluation
3,testeviario4.jpg,1149,648,qualitative stress test


External images selected for metrics: 3
Qualitative-only images: testeviario4.jpg


### 2. Criar ou revisar o ground truth de veiculos

Esta etapa prepara a avaliacao quantitativa sem alterar a configuracao final. Execute `preannotate_external_vehicles()` uma unica vez para salvar caixas iniciais do detector escolhido. Em seguida, consulte a lista exibida na celula anterior e execute `annotator = launch_external_vehicle_annotation("nome_da_imagem.jpg")` para cada arquivo.

Revise cada caixa: arraste para adicionar um veiculo ausente, clique com o botao direito dentro de uma caixa para remover falso positivo, use `d` para remover a ultima caixa e `s` para salvar. Considere um veiculo parcialmente ocluido quando ele ainda puder ser identificado; a caixa deve cobrir somente sua porcao visivel. As pre-anotacoes nao sao ground truth ate essa revisao humana.

In [32]:
external_images_by_name = {path.name: path for path in external_image_paths}


def launch_external_vehicle_annotation(file_name: str) -> ManualVehicleAnnotator:
    """Abre a imagem externa escolhida e recarrega caixas salvas, se existirem."""
    image_path = external_images_by_name.get(file_name)
    if image_path is None:
        available = ", ".join(external_images_by_name)
        raise ValueError(f"Arquivo nao encontrado. Opcoes: {available}")
    return launch_manual_annotation(image_path, external_annotation_path(image_path))


def preannotate_external_vehicles() -> None:
    """Salva caixas iniciais sem sobrescrever anotacoes ja revisadas."""
    EXTERNAL_ANNOTATIONS_DIR.mkdir(parents=True, exist_ok=True)
    detector = build_detector("yolo11n-obb.pt", "aerial", confidence=0.15, sahi_enabled=True)
    created = []
    skipped = []
    for image_path in external_image_paths:
        output_path = external_annotation_path(image_path)
        if output_path.exists():
            skipped.append(image_path.name)
            continue
        with Image.open(image_path) as external_image:
            external_array = np.asarray(external_image.convert("RGB"))
        detections = detector.predict(external_array)
        save_ground_truth(
            output_path,
            image_path,
            [detection.bbox_xyxy for detection in detections],
            annotation_method="model_assisted_preannotations_pending_manual_review",
            relative_to=ROOT,
        )
        created.append(f"{image_path.name}: {len(detections)} caixas")
    print(f"Pre-anotacoes criadas: {len(created)} | preservadas: {len(skipped)}")
    for row in created:
        print(row)


print(
    "Primeiro execute preannotate_external_vehicles(); "
    "depois revise cada arquivo com launch_external_vehicle_annotation(...)."
)

Primeiro execute preannotate_external_vehicles(); depois revise cada arquivo com launch_external_vehicle_annotation(...).


### 2. Executar veiculos e vias em lote

Execute antes a celula `neural-road-setup`, que carrega o segmentador. Esta celula cria uma unica instancia do detector de veiculos e a reutiliza para todas as imagens. Para cada arquivo, ela mede o tempo de cada tarefa, registra a quantidade de veiculos prevista e produz duas camadas de via: baseline HSV e Mask2Former `LABEL_3`. As deteccoes tambem ficam em memoria para a celula de metricas, sem executar nova inferencia.

A cobertura de via e apenas a porcentagem de pixels destacados, util para detectar mascaras exageradas. Ela nao mede acuracia sem uma mascara ground truth.

In [33]:
external_detector = build_detector("yolo11n-obb.pt", "aerial", confidence=0.15, sahi_enabled=True)
external_comparisons = []
external_rows = []

for image_path in external_image_paths:
    with Image.open(image_path) as external_image:
        external_array = np.asarray(external_image.convert("RGB"))

    started_at = perf_counter()
    external_detections = external_detector.predict(external_array)
    vehicle_inference_ms = (perf_counter() - started_at) * 1_000

    segmentation_inputs = segmentation_processor(
        images=Image.fromarray(external_array), return_tensors="pt"
    )
    segmentation_inputs = {
        name: value.to(SEGMENTATION_DEVICE) for name, value in segmentation_inputs.items()
    }
    started_at = perf_counter()
    with torch.inference_mode():
        segmentation_outputs = segmentation_model(**segmentation_inputs)
    road_inference_ms = (perf_counter() - started_at) * 1_000
    external_segmentation = (
        segmentation_processor.post_process_semantic_segmentation(
            segmentation_outputs, target_sizes=[external_array.shape[:2]]
        )[0]
        .detach()
        .cpu()
        .numpy()
    )
    neural_road_mask = (external_segmentation == SEGMENTATION_ROAD_CLASS_ID).astype(np.uint8) * 255
    neural_road_overlay = neural_overlay(external_array, neural_road_mask, opacity=0.40)

    hsv_roads = highlight_roads(
        external_array, hsv_lower=(5, 35, 45), hsv_upper=(35, 95, 190), kernel_size=7, min_area=1500
    )
    external_comparisons.append(
        {
            "file": image_path.name,
            "original": external_array,
            "detections": external_detections,
            "vehicles": draw_detections(external_array, external_detections),
            "hsv_roads": hsv_roads.overlay,
            "neural_roads": neural_road_overlay,
        }
    )
    external_rows.append(
        {
            "file": image_path.name,
            "predicted_vehicles": len(external_detections),
            "ground_truth_available": external_annotation_path(image_path).exists(),
            "vehicle_inference_ms": vehicle_inference_ms,
            "road_inference_ms": road_inference_ms,
            "hsv_road_coverage_pct": 100 * np.count_nonzero(hsv_roads.mask) / hsv_roads.mask.size,
            "neural_road_coverage_pct": 100
            * np.count_nonzero(neural_road_mask)
            / neural_road_mask.size,
        }
    )

external_results = pd.DataFrame(external_rows)
display(external_results.round(2))

,file,predicted_vehicles,ground_truth_available,vehicle_inference_ms,road_inference_ms,hsv_road_coverage_pct,neural_road_coverage_pct
0,testeviario1.jpg,19,True,381.39,106.58,0.98,17.78
1,testeviario2.jpg,25,True,311.61,106.11,6.92,3.02
2,testeviario3.jpg,50,True,380.29,109.90,2.80,15.70


### 4. Avaliar veiculos com ground truth revisado

Execute esta etapa somente depois de salvar as caixas de **todas** as imagens externas. A celula reutiliza as deteccoes da configuracao final ja executada na etapa anterior; portanto, as imagens externas avaliam generalizacao e nao participam de nova escolha de modelo, confidence ou slice size.

Cada imagem recebe TP, FP, FN, Precision, Recall, F1 e erro absoluto de contagem com IoU >= 0.50. O resumo usa metricas microagregadas: soma TP/FP/FN de todas as imagens antes de calcular Precision, Recall e F1. O erro absoluto medio de contagem permanece por imagem, sem permitir que erros opostos se cancelem.

In [34]:
from src.evaluation.ground_truth import load_ground_truth
from src.evaluation.metrics import calculate_detection_metrics

comparison_by_file = {comparison["file"]: comparison for comparison in external_comparisons}
missing_annotations = []
pending_annotations = []
for image_path in external_image_paths:
    annotation_path = external_annotation_path(image_path)
    if not annotation_path.exists():
        missing_annotations.append(image_path.name)
        continue
    payload = json.loads(annotation_path.read_text(encoding="utf-8"))
    if payload.get("annotation_method") != ANNOTATION_METHOD:
        pending_annotations.append(image_path.name)

if missing_annotations or pending_annotations:
    details = []
    if missing_annotations:
        details.append(f"sem JSON: {', '.join(missing_annotations)}")
    if pending_annotations:
        details.append(f"pendentes de revisao: {', '.join(pending_annotations)}")
    raise RuntimeError(
        "Nao calcule metricas com pre-anotacoes do modelo. "
        "Abra cada imagem com launch_external_vehicle_annotation(...), revise as caixas "
        f"e pressione s para salvar. Detalhes: {' | '.join(details)}"
    )

external_vehicle_metric_rows = []
external_vehicle_metric_comparisons = []
for image_path in external_image_paths:
    comparison = comparison_by_file[image_path.name]
    reviewed_ground_truth = load_ground_truth(external_annotation_path(image_path))
    metrics = calculate_detection_metrics(
        comparison["detections"], reviewed_ground_truth, iou_threshold=0.50
    )
    external_vehicle_metric_rows.append(
        {
            "file": image_path.name,
            "ground_truth_count": len(reviewed_ground_truth),
            "predicted_count": len(comparison["detections"]),
            "true_positives": metrics.true_positives,
            "false_positives": metrics.false_positives,
            "false_negatives": metrics.false_negatives,
            "precision": metrics.precision,
            "recall": metrics.recall,
            "f1": metrics.f1,
            "absolute_count_error": metrics.absolute_count_error,
        }
    )
    external_vehicle_metric_comparisons.append(
        {
            "file": image_path.name,
            "original": comparison["original"],
            "ground_truth": draw_ground_truth(comparison["original"], reviewed_ground_truth),
            "diagnostics": draw_detection_diagnostics(
                comparison["original"],
                comparison["detections"],
                reviewed_ground_truth,
                iou_threshold=0.50,
            ),
        }
    )

external_vehicle_metrics = pd.DataFrame(external_vehicle_metric_rows)
total_tp = int(external_vehicle_metrics["true_positives"].sum())
total_fp = int(external_vehicle_metrics["false_positives"].sum())
total_fn = int(external_vehicle_metrics["false_negatives"].sum())
micro_precision = total_tp / (total_tp + total_fp) if total_tp + total_fp else 0.0
micro_recall = total_tp / (total_tp + total_fn) if total_tp + total_fn else 0.0
micro_f1 = (
    2 * micro_precision * micro_recall / (micro_precision + micro_recall)
    if micro_precision + micro_recall
    else 0.0
)
external_vehicle_summary = pd.DataFrame(
    [
        {
            "images": len(external_vehicle_metrics),
            "ground_truth_vehicles": int(external_vehicle_metrics["ground_truth_count"].sum()),
            "predicted_vehicles": int(external_vehicle_metrics["predicted_count"].sum()),
            "true_positives": total_tp,
            "false_positives": total_fp,
            "false_negatives": total_fn,
            "micro_precision": micro_precision,
            "micro_recall": micro_recall,
            "micro_f1": micro_f1,
            "mean_absolute_count_error": external_vehicle_metrics["absolute_count_error"].mean(),
            "total_absolute_count_error": external_vehicle_metrics["absolute_count_error"].sum(),
        }
    ]
)

display(external_vehicle_metrics.round(3))
display(external_vehicle_summary.round(3))

,file,ground_truth_count,predicted_count,true_positives,false_positives,false_negatives,precision,recall,f1,absolute_count_error
0,testeviario1.jpg,20,19,19,0,1,1.00,0.95,0.974,1
1,testeviario2.jpg,25,25,25,0,0,1.00,1.00,1.000,0
2,testeviario3.jpg,50,50,49,1,1,0.98,0.98,0.980,0


,images,ground_truth_vehicles,predicted_vehicles,true_positives,false_positives,false_negatives,micro_precision,micro_recall,micro_f1,mean_absolute_count_error,total_absolute_count_error
0,3,95,94,93,1,2,0.989,0.979,0.984,0.333,1


### 5. Auditar visualmente as metricas de veiculos

Esta comparacao usa o mesmo matching das metricas: caixas verdes sao TP, amarelas sao FP e vermelhas sao FN. Ela torna possivel revisar se uma metrica baixa veio de oclusao, escala, confusao com objetos urbanos ou uma anotacao manual que precisa de correcao.

In [ ]:
figure, axes = plt.subplots(
    len(external_vehicle_metric_comparisons),
    3,
    figsize=(18, 5 * len(external_vehicle_metric_comparisons)),
)
axes = np.atleast_2d(axes)
for row_axes, comparison in zip(axes, external_vehicle_metric_comparisons, strict=True):
    views = [
        ("Original", comparison["original"]),
        ("Ground truth revisado", comparison["ground_truth"]),
        ("TP verde | FP amarelo | FN vermelho", comparison["diagnostics"]),
    ]
    for axis, (title, rendered) in zip(row_axes, views, strict=True):
        axis.imshow(rendered)
        axis.set_title(f"{comparison['file']} - {title}")
        axis.set_axis_off()
plt.tight_layout()
plt.show()

### 3. Revisar as comparacoes visuais

Para cada imagem, compare original, veiculos, HSV e Mask2Former. A leitura correta e: o segmentador neural melhora se preserva ruas e cruzamentos sem destacar telhados/solo como a mascara HSV. Para veiculos, use as metricas somente depois da revisao das caixas; antes disso, a contagem e apenas diagnostica. Registre exemplos bons e ruins no README apenas se a licenca das imagens permitir.

In [ ]:
figure, axes = plt.subplots(
    len(external_comparisons), 4, figsize=(22, 5 * len(external_comparisons))
)
axes = np.atleast_2d(axes)
for row_axes, comparison in zip(axes, external_comparisons, strict=True):
    views = [
        ("Original", comparison["original"]),
        ("Vehicle detections", comparison["vehicles"]),
        ("HSV road baseline", comparison["hsv_roads"]),
        ("Mask2Former road experiment", comparison["neural_roads"]),
    ]
    for axis, (title, rendered) in zip(row_axes, views, strict=True):
        axis.imshow(rendered)
        axis.set_title(f"{comparison['file']} — {title}")
        axis.set_axis_off()
plt.tight_layout()
plt.show()

## Analise composta: veiculos + malha viaria

Nesta etapa, a mesma imagem passa por dois modelos independentes: YOLO DOTA com SAHI localiza veiculos, e Mask2Former gera a mascara de via. Eles nao formam uma unica rede neural nem compartilham pesos; a uniao ocorre no pos-processamento. A mascara de via e aplicada primeiro, e as caixas de veiculo sao desenhadas por cima para preservar a leitura das deteccoes.

A execucao e sequencial de proposito: na GPU de 8 GB isso mantem o uso de memoria mais previsivel. O tempo total mostrado abaixo e a soma da deteccao, segmentacao e composicao. A camada de via oferece contexto visual e nao filtra deteccoes, pois veiculos tambem podem estar em estacionamentos, garagens e acessos privados.

### 1. Preparar a analise composta

Execute antes `benchmark-helpers` e `neural-road-setup`. A funcao abaixo reutiliza os modelos ja carregados e retorna as camadas separadas, o resultado combinado e os tempos. Assim, o experimento nao confunde uma composicao visual com uma nova metrica de qualidade.

In [37]:
def run_composed_analysis(image_array, detector):
    total_started_at = perf_counter()

    started_at = perf_counter()
    detections = detector.predict(image_array)
    vehicle_inference_ms = (perf_counter() - started_at) * 1_000

    segmentation_inputs = segmentation_processor(
        images=Image.fromarray(image_array), return_tensors="pt"
    )
    segmentation_inputs = {
        name: value.to(SEGMENTATION_DEVICE) for name, value in segmentation_inputs.items()
    }
    started_at = perf_counter()
    with torch.inference_mode():
        segmentation_outputs = segmentation_model(**segmentation_inputs)
    road_inference_ms = (perf_counter() - started_at) * 1_000

    segmentation = (
        segmentation_processor.post_process_semantic_segmentation(
            segmentation_outputs, target_sizes=[image_array.shape[:2]]
        )[0]
        .detach()
        .cpu()
        .numpy()
    )
    road_mask = (segmentation == SEGMENTATION_ROAD_CLASS_ID).astype(np.uint8) * 255
    road_overlay = neural_overlay(image_array, road_mask, opacity=0.40)

    return {
        "detections": detections,
        "road_mask": road_mask,
        "vehicle_overlay": draw_detections(image_array, detections),
        "road_overlay": road_overlay,
        "combined_overlay": draw_detections(road_overlay, detections),
        "vehicle_inference_ms": vehicle_inference_ms,
        "road_inference_ms": road_inference_ms,
        "total_analysis_ms": (perf_counter() - total_started_at) * 1_000,
    }

### 2. Testar na imagem do desafio

Esta celula utiliza exatamente a configuracao final de veiculos: `yolo11n-obb.pt`, dominio aereo, confidence 0.15 e SAHI com slices de 512 px. As metricas de veiculos podem continuar sendo calculadas separadamente contra o ground truth existente; para a via, esta tela e uma revisao visual, porque ainda nao ha mascara de referencia.

In [ ]:
composed_detector = build_detector("yolo11n-obb.pt", "aerial", confidence=0.15, sahi_enabled=True)
challenge_composed_result = run_composed_analysis(image_array, composed_detector)

figure, axes = plt.subplots(1, 4, figsize=(26, 7))
views = [
    ("Original", image_array),
    ("Vehicle detections", challenge_composed_result["vehicle_overlay"]),
    ("Mask2Former road layer", challenge_composed_result["road_overlay"]),
    ("Combined analysis", challenge_composed_result["combined_overlay"]),
]
for axis, (title, rendered) in zip(axes, views, strict=True):
    axis.imshow(rendered)
    axis.set_title(title)
    axis.set_axis_off()
plt.tight_layout()
plt.show()
print(f"Predicted vehicles: {len(challenge_composed_result['detections'])}")
print(f"Vehicle inference: {challenge_composed_result['vehicle_inference_ms']:.1f} ms")
print(f"Road inference: {challenge_composed_result['road_inference_ms']:.1f} ms")
print(f"Total composed analysis: {challenge_composed_result['total_analysis_ms']:.1f} ms")

### 3. Revisar a analise composta em imagens externas

Execute antes `external-image-discovery`, `external-batch-inference` e a celula de metricas de veiculos externos. A tabela resultante apresenta, para cada imagem, TP, FP, FN, Precision, Recall, F1, erro absoluto de contagem e os tempos de veiculos, vias e analise total.

As caixas de veiculos sao ciano e a malha viaria e laranja. Esta comparacao avalia generalizacao; ela nao reabre a escolha de modelo, confidence ou slice size.

In [39]:
if "external_vehicle_metrics" not in globals():
    raise RuntimeError(
        "Execute primeiro a celula de metricas de veiculos externos. "
        "Ela usa os ground truths revisados para montar este comparativo."
    )

external_composed_results = []
external_composed_rows = []

for image_path in external_image_paths:
    with Image.open(image_path) as external_image:
        external_array = np.asarray(external_image.convert("RGB"))

    result = run_composed_analysis(external_array, composed_detector)
    external_composed_results.append(
        {
            "file": image_path.name,
            "original": external_array,
            "combined": result["combined_overlay"],
        }
    )
    external_composed_rows.append(
        {
            "file": image_path.name,
            "predicted_vehicles": len(result["detections"]),
            "vehicle_inference_ms": result["vehicle_inference_ms"],
            "road_inference_ms": result["road_inference_ms"],
            "total_analysis_ms": result["total_analysis_ms"],
        }
    )

external_composed_timing = pd.DataFrame(external_composed_rows)
external_composed_metrics = external_vehicle_metrics.merge(
    external_composed_timing, on="file", validate="one_to_one"
)
external_composed_metrics = external_composed_metrics[
    [
        "file",
        "ground_truth_count",
        "predicted_count",
        "true_positives",
        "false_positives",
        "false_negatives",
        "precision",
        "recall",
        "f1",
        "absolute_count_error",
        "vehicle_inference_ms",
        "road_inference_ms",
        "total_analysis_ms",
    ]
]
display(external_composed_metrics.round(3))

,file,ground_truth_count,predicted_count,true_positives,false_positives,false_negatives,precision,recall,f1,absolute_count_error,vehicle_inference_ms,road_inference_ms,total_analysis_ms
0,testeviario1.jpg,20,19,19,0,1,1.00,0.95,0.974,1,335.747,124.155,503.938
1,testeviario2.jpg,25,25,25,0,0,1.00,1.00,1.000,0,387.108,116.689,540.165
2,testeviario3.jpg,50,50,49,1,1,0.98,0.98,0.980,0,353.286,110.420,509.358


### 4. Visualizar o resultado unificado

Cada linha mostra a imagem original e a analise composta. O titulo resume TP, FP, FN, F1 e erro de contagem da mesma imagem; a tabela anterior contem todos os valores. Revise se (1) as caixas continuam legiveis em ciano sobre a camada de via laranja; (2) a via adiciona contexto sem cobrir a cena inteira; e (3) nenhum resultado foi usado para ajustar parametros especificamente nestas imagens.

In [ ]:
figure, axes = plt.subplots(
    len(external_composed_results), 2, figsize=(16, 5 * len(external_composed_results))
)
axes = np.atleast_2d(axes)
metrics_by_file = external_composed_metrics.set_index("file").to_dict(orient="index")
for row_axes, result in zip(axes, external_composed_results, strict=True):
    metrics = metrics_by_file[result["file"]]
    metric_summary = (
        f"Analise composta | TP {int(metrics['true_positives'])} | "
        f"FP {int(metrics['false_positives'])} | FN {int(metrics['false_negatives'])}\n"
        f"F1 {metrics['f1']:.3f} | erro de contagem {int(metrics['absolute_count_error'])}"
    )
    views = [("Original", result["original"]), (metric_summary, result["combined"])]
    for axis, (title, rendered) in zip(row_axes, views, strict=True):
        axis.imshow(rendered)
        axis.set_title(f"{result['file']} - {title}")
        axis.set_axis_off()
plt.tight_layout()
plt.show()

## Registro da decisao

Os resultados executados nesta imagem estao registrados em `outputs/metrics/benchmark_results.json`. A configuracao final so deve ser atualizada depois de comparar F1, recall, erro absoluto de contagem e tempo. Como o ground truth foi iniciado por pre-anotacoes do modelo aereo e revisado manualmente, os resultados sao evidencias para esta prova, nao uma estimativa de generalizacao para novas imagens.